In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sqlite3
from glob import glob

import joblib
import pandas as pd
import requests
from arch.univariate.base import ARCHModelResult
from config import settings
from data import SQLRepository
from IPython.display import VimeoVideo


In [ ]:
from mock_alpha import activate_mock
activate_mock()


In [ ]:
VimeoVideo("772219745", h="f3bfda20cd", width=600)


In [ ]:
VimeoVideo("772219717", h="8f1afa7919", width=600)


In [ ]:
connection = sqlite3.connect(settings.db_name,check_same_thread=False)
repo = SQLRepository(connection=connection)

print("repo type:", type(repo))
print("repo.connection type:", type(repo.connection))


In [ ]:
VimeoVideo("772219669", h="1d225ab776", width=600)


In [ ]:
from model import GarchModel

# Instantiate a `GarchModel`
gm_ambuja = GarchModel(ticker="AMBUJACEM.BSE", repo=repo, use_new_data=False)

# Does `gm_ambuja` have the correct attributes?
assert gm_ambuja.ticker == "AMBUJACEM.BSE"
assert gm_ambuja.repo == repo
assert not gm_ambuja.use_new_data
assert gm_ambuja.model_directory == settings.model_directory


In [ ]:
VimeoVideo("772219593", h="3f3c401c04", width=600)


In [ ]:
# Instantiate `GarchModel`, use new data
model_shop = GarchModel(ticker="SHOPERSTOP.NS", repo=repo, use_new_data=True)

# Check that model doesn't have `data` attribute yet
assert not hasattr(model_shop, "data")

# Wrangle data
model_shop.wrangle_data(n_observations=1000)

# Does model now have `data` attribute?
assert hasattr(model_shop, "data")

# Is the `data` a Series?
assert isinstance(model_shop.data, pd.Series)

# Is Series correct shape?
assert model_shop.data.shape == (1000,)

model_shop.data.head()


In [ ]:
VimeoVideo("772219535", h="55fbfdff55", width=600)


In [ ]:
# Instantiate `GarchModel`, use old data
model_shop = GarchModel(ticker="SHOPERSTOP.NS", repo=repo, use_new_data=False)

# Wrangle data
model_shop.wrangle_data(n_observations=1000)

# Fit GARCH(1,1) model to data
model_shop.fit(p=1, q=1)

# Does `model_shop` have a `model` attribute now?
assert hasattr(model_shop, "model")

# Is model correct data type?
assert isinstance(model_shop.model, ARCHModelResult)

# Does model have correct parameters?
assert model_shop.model.params.index.tolist() == ["mu", "omega", "alpha[1]", "beta[1]"]

# Check model parameters
model_shop.model.summary()


In [ ]:
VimeoVideo("772219489", h="3de8abb0e6", width=600)


In [ ]:
# Generate prediction from `model_shop`
prediction = model_shop.predict_volatility(horizon=5)

# Is prediction a dictionary?
assert isinstance(prediction, dict)

# Are keys correct data type?
assert all(isinstance(k, str) for k in prediction.keys())

# Are values correct data type?
assert all(isinstance(v, float) for v in prediction.values())

prediction


In [ ]:
VimeoVideo("772219427", h="0dd5731a0d", width=600)


In [ ]:
# Save `model_shop` model, assign filename
filename = model_shop.dump()

# Is `filename` a string?
assert isinstance(filename, str)

# Does filename include ticker symbol?
assert model_shop.ticker in filename

# Does file exist?
assert os.path.exists(filename)

filename


In [ ]:
VimeoVideo("772219326", h="4e1f9421e4", width=600)


In [ ]:
def load(ticker):
    """Load latest model from model directory.

    Parameters
    ----------
    ticker : str
        Ticker symbol for which model was trained.

    Returns
    -------
    `ARCHModelResult`
    """
    # Create pattern for glob search
    pattern = os.path.join(settings.model_directory, f"*{ticker}.pkl")

    # Try to find path of latest model
    try:
        model_path = sorted(glob(pattern))[-1]
    except IndexError:
        raise Exception(f"No model trained for '{ticker}'.")
    # Handle possible `IndexError`
    
    # Load model
    model = joblib.load(model_path)

    # Return model
    return model


In [ ]:
# Assign load output to `model`
model_shop = load(ticker="SHOPERSTOP.NS")

# Does function return an `ARCHModelResult`
assert isinstance(model_shop, ARCHModelResult)

# Check model parameters
model_shop.summary()


In [ ]:
VimeoVideo("772219392", h="deed99bf85", width=600)


In [ ]:
model_shop = GarchModel(ticker="SHOPERSTOP.NS", repo=repo, use_new_data=False)

# Check that new `model_shop_test` doesn't have model attached
assert not hasattr(model_shop, "model")

# Load model
model_shop.load()

# Does `model_shop_test` have model attached?
assert hasattr(model_shop, "model")

model_shop.model.summary()


In [ ]:
VimeoVideo("772219283", h="2cd1d97516", width=600)


In [ ]:
VimeoVideo("772219237", h="5ee74f82db", width=600)


In [ ]:
uvicorn main:app --reload --workers 1 --host localhost --port 8008


In [ ]:
VimeoVideo("772219175", h="6f53c61020", width=600)


In [ ]:
VimeoVideo("772219134", h="09a4b98413", width=600)


In [ ]:
url = "http://localhost:8008/hello"

response = requests.get(url=url)

print("response code:", response.status_code)
response.json()


In [ ]:
VimeoVideo("772219078", h="4f016b11e1", width=600)


In [ ]:
VimeoVideo("772219008", h="ad1114eb9e", width=600)


In [ ]:
from main import FitIn, FitOut

# Instantiate `FitIn`. Play with parameters.
fi = FitIn(
    ticker="SHOPERSTOP.BSE",
    use_new_data=True,
    n_observations=2000,
    p=1,
    q=1
)
print(fi)
# Instantiate `FitOut`. Play with parameters.
fo = FitOut(
    ticker="SHOPERSTOP.BSE",
    use_new_data=True,
    n_observations=2000,
    p=1,
    q=1,
    success=True,
    message="Model is ready to rock!!!"
)
print(fo)


In [ ]:
VimeoVideo("772218958", h="37744c9d88", width=600)


In [ ]:
from main import build_model

# Instantiate `GarchModel` with function
model_shop = build_model(ticker="SHOPERSTOP.NS", use_new_data=False)

# Is `SQLRepository` attached to `model_shop`?
assert isinstance(model_shop.repo, SQLRepository)

# Is SQLite database attached to `SQLRepository`
assert isinstance(model_shop.repo.connection, sqlite3.Connection)

# Is `ticker` attribute correct?
assert model_shop.ticker == "SHOPERSTOP.NS"

# Is `use_new_data` attribute correct?
assert not model_shop.use_new_data

model_shop


In [ ]:
VimeoVideo("772218892", h="6779ee3470", width=600)


In [ ]:
VimeoVideo("772218833", h="6d27fb4539", width=600)


In [ ]:
# URL of `/fit` path
url = "http://localhost:8008/fit"

# Data to send to path
json = {
    "ticker": "SHOPERSTOP.BSE",
    "use_new_data": False,
    "n_observations": 2000,
    "p": 1,
    "q": 1
}

# Response of POST request
response = requests.post(url=url, json=json)

# Inspect response
print("response code:", response.status_code)
response.json()


In [ ]:
VimeoVideo("772218808", h="3a73624069", width=600)


In [ ]:
from main import PredictIn, PredictOut

pi = PredictIn(ticker="SHOPERSTOP.NS", n_days=5)
print(pi)

po = PredictOut(
    ticker="SHOPERSTOP.NS", n_days=5, success=True, forecast={}, message="success"
)
print(po)


In [ ]:
VimeoVideo("772218740", h="ff06859ece", width=600)


In [ ]:
VimeoVideo("772218642", h="1da744b9e7", width=600)


In [ ]:
# URL of `/predict` path
url = "http://localhost:8008/predict"

# Data to send to path
json = {"ticker": "SHOPERSTOP.BSE","n_days": 5}

# Response of POST request
response = requests.post(url=url, json=json)

# Response JSON to be submitted to grader
submission = response.json()

# Inspect JSON
submission
